# Developer Salary Prediction: Multi-Class Ordinal Classification

**Dataset:** Kaggle Machine Learning & Data Science Survey 2022 — global respondents, cleaned subset with compensation buckets as target.

**Objective:** Predict a survey respondent's annual compensation bucket (ordinal classes) from survey responses about tools, experience, and background.

**Pipeline:** Data cleaning → feature engineering → Chi-square + LassoCV selection → ordinal logistic regression → k-fold CV → GridSearchCV tuning → test evaluation

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.metrics import accuracy_score, classification_report
from scipy import stats

In [ ]:
salaries_raw = pd.read_csv("data/clean_kaggle_data_2022.csv", low_memory = False, encoding = 'latin2')
salaries = pd.read_csv("data/clean_kaggle_data_2022.csv", low_memory = False, encoding = 'latin2')
salaries.shape

In [ ]:
# Set the display option to show all columns
pd.set_option('display.max_columns', None)

salaries.head()

In [ ]:
pd.set_option('display.max_colwidth', None)  # Set to None to show the entire content
salaries.head(1)

In [ ]:
# Determine all the unique data points in the column Q44_7 of the df salaries_raw in a list

unique_q44_7 = salaries_raw['Q23'].unique().tolist()
unique_q44_7

## 1. Data Cleaning

In [ ]:
def CleanData(df):
    # Drop the first row (question details)
    df.drop(df.index[0], inplace=True)

    # Fill missing values for numeric columns with the median or mean
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in numeric_cols:
        df[col].fillna(df[col].median(), inplace=True)

    # Drop irrelevant columns, including ones specified based on analysis
    irrelevant_columns = [
    "Q6_1", "Q6_2", "Q6_3", "Q6_4", "Q6_5", "Q6_6", "Q6_7", "Q6_8", "Q6_9", "Q6_10", "Q6_11", "Q6_12",
    "Q7_1", "Q7_2", "Q7_3", "Q7_4", "Q7_5", "Q7_6", "Q7_7",
    "Q10_1", "Q10_2", "Q10_3",
    "Q13_1", "Q13_2", "Q13_3", "Q13_4", "Q13_5", "Q13_6", "Q13_7", "Q13_8", "Q13_9", "Q13_10", "Q13_11", "Q13_12", "Q13_13", "Q13_14",
    "Q14_1", "Q14_2", "Q14_3", "Q14_4", "Q14_5", "Q14_6", "Q14_7", "Q14_8", "Q14_9", "Q14_10", "Q14_11", "Q14_12", "Q14_13", "Q14_14", "Q14_15", "Q14_16",
    "Q15_1", "Q15_2", "Q15_3", "Q15_4", "Q15_5", "Q15_6", "Q15_7", "Q15_8", "Q15_9", "Q15_10", "Q15_11", "Q15_12", "Q15_13", "Q15_14", "Q15_15",
    "Q17_1", "Q17_2", "Q17_3", "Q17_4", "Q17_5", "Q17_6", "Q17_7", "Q17_8", "Q17_9", "Q17_10", "Q17_11", "Q17_12", "Q17_13", "Q17_14", "Q17_15",
    "Q18_1", "Q18_2", "Q18_3", "Q18_4", "Q18_5", "Q18_6", "Q18_7", "Q18_8", "Q18_9", "Q18_10", "Q18_11", "Q18_12", "Q18_13", "Q18_14",
    "Q20_1", "Q20_2", "Q20_3", "Q20_4", "Q20_5", "Q20_6",
    "Q21_1", "Q21_2", "Q21_3", "Q21_4", "Q21_5", "Q21_6", "Q21_7", "Q21_8", "Q21_9", "Q21_10",
    "Q22",
    "Q26",
    "Q28_1", "Q28_2", "Q28_3", "Q28_4", "Q28_5", "Q28_6", "Q28_7", "Q28_8",
    "Q31_1", "Q31_2", "Q31_3", "Q31_4", "Q31_5", "Q31_6", "Q31_7", "Q31_8", "Q31_9", "Q31_10", "Q31_11", "Q31_12",
    "Q32",
    "Q40_1", "Q40_2", "Q40_3", "Q40_4", "Q40_5", "Q40_6", "Q40_7", "Q40_8", "Q40_9", "Q40_10", "Q40_11", "Q40_12", "Q40_13", "Q40_14", "Q40_15",
    "Q41_1", "Q41_2", "Q41_3", "Q41_4", "Q41_5", "Q41_6", "Q41_7", "Q41_8", "Q41_9",
    "Q43",
    "Q44_1", "Q44_2", "Q44_3", "Q44_4", "Q44_5", "Q44_6", "Q44_7", "Q44_8", "Q44_9", "Q44_10", "Q44_11", "Q44_12"
]
    df.drop(columns=irrelevant_columns, inplace=True, errors='ignore')

    return df

# Apply the cleaning function
salaries = CleanData(salaries)
salaries.shape

In [ ]:
salaries.head()

In [ ]:
salaries.shape

### 1.1 Impute Missing Values

In [ ]:
def ImputingSingleColMissingValues(df):
    # Define the list of single column names (including Q29_Encoded, the target column)
    single_col_names = ["Duration (in seconds)", "Q2", "Q3", "Q4", "Q5", "Q8", "Q9", "Q11", "Q16", "Q23", "Q24", "Q25", "Q27", "Q29", "Q29_Encoded", "Q30", "Q29_Encoded" ] # Including Q29_Encoded as specified

    # Display the percentage of null values in the single column responses
    print("Percentage of null values:")
    print(df[single_col_names].isnull().sum() * 100 / len(df))

    # Address missing values in single column responses
    for col in single_col_names:
        if df[col].dtype in ['float64', 'int64']:  # For numeric columns
            df[col].fillna(df[col].median(), inplace=True)  # Fill with median
        else:  # For categorical columns
            df[col].fillna("Unknown", inplace=True)  # Fill with "Unknown" placeholder

    # Verify missing values are addressed
    print("Percentage of null values (make sure they are all zeros):")
    print(df[single_col_names].isnull().sum() * 100 / len(df))

    # Assert to ensure no missing values remain
    assert df[single_col_names].isnull().values.sum() == 0, \
        "There are still missing values remaining!"

    return df

# Apply the function to the 'salaries' DataFrame
salaries = ImputingSingleColMissingValues(salaries)

### 1.2 Encode Categorical Features

In [ ]:
def EncodeCategoricalFeatures(df, columns):
    label_encoder = LabelEncoder()
    for col in columns:
        if df[col].dtype == 'object':  # Check if the column is categorical
            df[col] = label_encoder.fit_transform(df[col].astype(str))  # Label encode categorical columns
    return df

# List of single-response columns to encode, including only categorical ones
single_cols = ["Duration (in seconds)", "Q2", "Q3", "Q4", "Q5", "Q8", "Q9", "Q11", "Q16", "Q23", "Q24", "Q25", "Q27", "Q29", "Q29_Encoded", "Q30", "Q29_Encoded"]

# Apply the encoding function
salaries = EncodeCategoricalFeatures(salaries, single_cols)

In [ ]:
salaries.head()

In [ ]:
multi_col_names_raw = [
    "Q12_1", "Q12_2", "Q12_3", "Q12_4", "Q12_5", "Q12_6", "Q12_7", "Q12_8", "Q12_9", "Q12_10", "Q12_11", "Q12_12", "Q12_13", "Q12_14", "Q12_15", # Q12 - Programming languages
    "Q19_1", "Q19_2", "Q19_3", "Q19_4", "Q19_5", "Q19_6", "Q19_7", "Q19_8",  # Q19 - Computer vision methods
    "Q33_1", "Q33_2", "Q33_3", "Q33_4", "Q33_5", # Q33 - Machine learning activities at work
    "Q34_1", "Q34_2", "Q34_3", "Q34_4", "Q34_5", "Q34_6", "Q34_7", "Q34_8", # Q34 - Job roles
    "Q35_1", "Q35_2", "Q35_3", "Q35_4", "Q35_5", "Q35_6", "Q35_7", "Q35_8", "Q35_9", "Q35_10", "Q35_11", "Q35_12", "Q35_13", "Q35_14", "Q35_15", "Q35_16", # Q35 - Industry sectors
    "Q36_1", "Q36_2", "Q36_3", "Q36_4", "Q36_5", "Q36_6", "Q36_7", "Q36_8", "Q36_9", "Q36_10", "Q36_11", "Q36_12", "Q36_13", "Q36_14", "Q36_15", # Q36 - Company sizes
    "Q37_1", "Q37_2", "Q37_3", "Q37_4", "Q37_5", "Q37_6", "Q37_7", "Q37_8", "Q37_9", "Q37_10", "Q37_11", "Q37_12", "Q37_13", # Q37 - Data storage products
    "Q38_1", "Q38_2", "Q38_3", "Q38_4", "Q38_5", "Q38_6", "Q38_7", "Q38_8", # Q38 - Business intelligence tools
    "Q39_1", "Q39_2", "Q39_3", "Q39_4", "Q39_5", "Q39_6", "Q39_7", "Q39_8", "Q39_9", "Q39_10", "Q39_11", "Q39_12", # Q39 - Managed machine learning products
    "Q42_1", "Q42_2", "Q42_3", "Q42_4", "Q42_5", "Q42_6", "Q42_7", "Q42_8", "Q42_9" # Q42 - Monitoring tools for ML models
    ]


def ProcessMultiColumnResponses(df):
    # List of multi-column responses
    multi_col_names = [
    "Q12_1", "Q12_2", "Q12_3", "Q12_4", "Q12_5", "Q12_6", "Q12_7", "Q12_8", "Q12_9", "Q12_10", "Q12_11", "Q12_12", "Q12_13", "Q12_14", "Q12_15", # Q12 - Programming languages
    "Q19_1", "Q19_2", "Q19_3", "Q19_4", "Q19_5", "Q19_6", "Q19_7", "Q19_8",  # Q19 - Computer vision methods
    "Q33_1", "Q33_2", "Q33_3", "Q33_4", "Q33_5", # Q33 - Machine learning activities at work
    "Q34_1", "Q34_2", "Q34_3", "Q34_4", "Q34_5", "Q34_6", "Q34_7", "Q34_8", # Q34 - Job roles
    "Q35_1", "Q35_2", "Q35_3", "Q35_4", "Q35_5", "Q35_6", "Q35_7", "Q35_8", "Q35_9", "Q35_10", "Q35_11", "Q35_12", "Q35_13", "Q35_14", "Q35_15", "Q35_16", # Q35 - Industry sectors
    "Q36_1", "Q36_2", "Q36_3", "Q36_4", "Q36_5", "Q36_6", "Q36_7", "Q36_8", "Q36_9", "Q36_10", "Q36_11", "Q36_12", "Q36_13", "Q36_14", "Q36_15", # Q36 - Company sizes
    "Q37_1", "Q37_2", "Q37_3", "Q37_4", "Q37_5", "Q37_6", "Q37_7", "Q37_8", "Q37_9", "Q37_10", "Q37_11", "Q37_12", "Q37_13", # Q37 - Data storage products
    "Q38_1", "Q38_2", "Q38_3", "Q38_4", "Q38_5", "Q38_6", "Q38_7", "Q38_8", # Q38 - Business intelligence tools
    "Q39_1", "Q39_2", "Q39_3", "Q39_4", "Q39_5", "Q39_6", "Q39_7", "Q39_8", "Q39_9", "Q39_10", "Q39_11", "Q39_12", # Q39 - Managed machine learning products
    "Q42_1", "Q42_2", "Q42_3", "Q42_4", "Q42_5", "Q42_6", "Q42_7", "Q42_8", "Q42_9" # Q42 - Monitoring tools for ML models
          ]

    # Convert all responses to strings to standardize data types
    df[multi_col_names] = df[multi_col_names].astype(str)

    # Fill missing values with "Unknown" placeholder
    df[multi_col_names] = df[multi_col_names].fillna("Unknown")

    # Perform one-hot encoding or label encoding as needed
    df = pd.get_dummies(df, columns=multi_col_names, drop_first=True) #.astype(int)

    print("Missing values handled and categorical encoding completed for multi-column responses.")

    return df

# Apply the processing function
salaries = ProcessMultiColumnResponses(salaries)



In [ ]:
salaries.head()

In [ ]:
# Convert all boolean columns to integers (0 and 1)
bool_cols = salaries.select_dtypes(include=['bool']).columns
salaries[bool_cols] = salaries[bool_cols].astype(int)

print("Boolean columns converted to integers (0 and 1).")

In [ ]:

# Convert 'Q29_Encoded' to integer type
salaries['Q29_Encoded'] = salaries['Q29_Encoded'].astype(int)

In [ ]:
salaries.head()

### 1.3 Define Target Variable

In [ ]:
# Make sure there are no missing values remaining in the dataset
assert salaries.isnull().values.sum() == 0, \
    "There are still {} missing values remaining in salaries!".format(
        salaries.isnull().values.sum()
    )

In [ ]:
## Define the target variable
target = salaries["Q29_Encoded"]  # Assuming Q29_Encoded is the target


# Drop the target variable from the feature set
salaries.drop(columns=["Q29_Encoded"], inplace=True)
salaries.drop(columns=["Q29"], inplace=True)
salaries.drop(columns=["Q29_buckets"], inplace=True)

# Make sure the target variables are not included in the feature set
for col in salaries.columns:
    assert 'Q29' not in col, \
        "Target variable ({}) is still in the dataset".format(col)

## 2. Feature Analysis & Selection

### 2.1 Train/Test Split

In [ ]:
# Assuming 'salaries' is your DataFrame and 'Q29_encoded' is your target variable
X = salaries  # Feature set
y = target    # Target variable

# Perform the train-test split
train_df, test_df, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training and test sets created.")
print("Training set shape:", train_df.shape, y_train.shape)
print("Test set shape:", test_df.shape, y_test.shape)

### 2.2 Feature Engineering

In [ ]:
# Basic statistics for the training data
print(train_df.describe())

# Count of missing values per column for the training data
print(train_df.isnull().sum())

In [ ]:
# Plotting all feature-target count plots


# Define the target column
target_col = 'Q29_Encoded'

# Define the number of plots per row
plots_per_row = 3

# Get a list of columns excluding the target column
columns = [col for col in visualization_df.columns if col != target_col]

# Calculate the number of rows needed
num_rows = math.ceil(len(columns) / plots_per_row)

# Create the figure for subplots
fig, axes = plt.subplots(num_rows, plots_per_row, figsize=(15, num_rows * 5))
axes = axes.flatten()  # Flatten to iterate easily

# Loop through each column and plot
for idx, col in enumerate(columns):
    sns.countplot(data=visualization_df, x=col, hue=target_col, ax=axes[idx])
    axes[idx].set_title(f'Count Plot of {col} vs. {target_col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].legend(title=target_col)
    axes[idx].tick_params(axis='x', rotation=45)  # Rotate x labels if needed for better readability

# Turn off any empty subplots if the number of columns is not a multiple of plots_per_row
for idx in range(len(columns), len(axes)):
    fig.delaxes(axes[idx])

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# Chi-square analysis for all the features in the dataset, order from highest to lowest statistical significance

import pandas as pd
import scipy.stats as stats
from IPython.display import display  # For nice display in Jupyter

# Assume `visualization_df` is your DataFrame with features and the target variable `Q29_Encoded`
# List of categorical features (excluding Q29_Encoded, the target variable itself)
categorical_features = [col for col in visualization_df.columns if col != 'Q29_Encoded']

# Significance level
alpha = 0.05

# Results list to store each feature's results
results = []

# Loop over each categorical feature and perform Chi-Square test
for feature in categorical_features:
    # Create a contingency table
    contingency_table = pd.crosstab(visualization_df[feature], visualization_df['Q29_Encoded'])

    # Perform the Chi-Square test
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

    # Determine if the result is statistically significant
    significant = 'Yes' if p_value < alpha else 'No'

    # Append results to the list
    results.append({
        'Feature': feature,
        'Chi-Square Statistic': round(chi2, 2),
        'p-Value': round(p_value, 4),
        'Degrees of Freedom': dof,
        'Statistically Significant': significant
    })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Sort by p-value (ascending) to show most statistically significant features first
results_df = results_df.sort_values(by='p-Value', ascending=True)

# Style the DataFrame for a nicer display
styled_results_df = results_df.style.set_properties(**{
    'background-color': 'white',
    'color': 'black',
    'border-color': 'black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('color', 'black')]},
    {'selector': 'tr:hover', 'props': [('background-color', '#d1e7fd')]}
]).set_caption("Chi-Square Test Results for Categorical Features (Ordered by Significance)")

# Display the styled DataFrame
display(styled_results_df)


In [ ]:

# Assuming `results_df` is the DataFrame you created in the previous code

# Display all rows of the DataFrame
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
  display(results_df)

In [ ]:

# 1. Combining experience and eduction level
# More experienced individuals with higher education levels may have higher compensation
train_df['Experience_Education'] = train_df['Q2'] * train_df['Q3']  # Example combination

# 2. Combining skill related features. The more skills a respondent has, the higher their potential earning power might be.
#This feature could capture a cumulative effect of skills on compensation.
train_df['Total_Skills'] = train_df[['Q12_1_nan', 'Q12_2_nan', 'Q12_3_nan', 'Q12_4_nan', 'Q12_5_nan', 'Q12_6_nan', 'Q12_7_nan', 'Q12_8_nan', 'Q12_9_nan', 'Q12_10_nan', 'Q12_11_nan', 'Q12_12_nan', 'Q12_13_nan', 'Q12_15_nan']].sum(axis=1)



In [ ]:
train_df.head()

### 2.3 Lasso Feature Selection

In [ ]:
# Step 1: Prepare your data
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_df)

# Step 2: Initialize and fit Lasso regression with cross-validation to find the best alpha (regularization strength)
lasso = LassoCV(cv=5, random_state=0)  # cv=5 for 5-fold cross-validation
lasso.fit(X_scaled, y_train)

# Step 3: Display the coefficients
feature_names = train_df.columns
lasso_coefs = pd.Series(lasso.coef_, index=feature_names)

# Filter out features with non-zero coefficients
selected_features = lasso_coefs[lasso_coefs != 0].index.tolist()

# Display the selected features
print("Selected features from Lasso Regression:")
print(selected_features)
print("Lenght of list:", len(selected_features))

# Optionally, display a table of all features with their coefficients
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso.coef_
}).sort_values(by='Coefficient', ascending=False)
print("Length of table:", len(feature_importance_df))


# Filter features with coefficients greater than 0.09 or less than -0.09
significant_features_df = feature_importance_df[(feature_importance_df['Coefficient'] > 0.09) | (feature_importance_df['Coefficient'] < -0.09)]

# Display the filtered DataFrame
print("Features with coefficient > 0.05 or < -0.05:")
print(significant_features_df)

# Optionally, extract just the feature names to a list
significant_features = significant_features_df['Feature'].tolist()
print("Selected significant features:", significant_features)
print("Length of significant features list:", len(significant_features))


# Set display options to show the entire DataFrame
pd.set_option('display.max_rows', None)
print("Feature Importance Table:")
print(feature_importance_df)
pd.reset_option('display.max_rows')  # Reset to default after display




# Plot all feature coefficients
plt.figure(figsize=(10, 8))
lasso_coefs.plot(kind='bar', color='teal')
plt.title("Feature Importance Based on Lasso Coefficients")
plt.xlabel("Features")
plt.ylabel("Coefficient")
plt.xticks(rotation=90, fontsize=8)  # Set fontsize to a smaller size
plt.show()

# Optional: Plot only significant features (if needed)
plt.figure(figsize=(10, 8))
significant_features_df.set_index('Feature')['Coefficient'].plot(kind='bar', color='coral')
plt.title("Significant Feature Importance Based on Lasso Coefficients")
plt.xlabel("Features")
plt.ylabel("Coefficient")
plt.xticks(rotation=90)
plt.show()


In [ ]:
selected_features = ['Q4', 'Q2', 'Q27', 'Q35_1_nan', 'Q12_10_nan', 'Q36_2_nan', 'Q36_3_nan', 'Q24', 'Q12_11_nan', 'Q30', 'Q12_4_nan', 'Q36_11_nan', 'Q37_8_nan', 'Q12_7_nan', 'Q42_1_nan', 'Q35_13_nan', 'Q39_3_nan', 'Q11', 'Q36_9_nan', 'Q12_2_nan', 'Q35_16_nan', 'Q12_3_nan', 'Q35_10_nan', 'Experience_Education', 'Q35_6_nan', 'Q12_9_nan', 'Q34_3_nan']

In [ ]:
train_df_selected = train_df[selected_features]

In [ ]:
train_df_selected.head()

### 2.4 Apply to Test Data

In [ ]:

# 1. Combining experience and eduction level. More experienced individuals with higher education levels may have higher compensation
test_df['Experience_Education'] = test_df['Q2'] * test_df['Q3']  # Example combination

# 2. Combining skill related features. The more skills a respondent has, the higher their potential earning power might be.
test_df['Total_Skills'] = test_df[['Q12_1_nan', 'Q12_2_nan', 'Q12_3_nan', 'Q12_4_nan', 'Q12_5_nan', 'Q12_6_nan', 'Q12_7_nan', 'Q12_8_nan', 'Q12_9_nan', 'Q12_10_nan', 'Q12_11_nan', 'Q12_12_nan', 'Q12_13_nan', 'Q12_15_nan']].sum(axis=1)


# Leave selected features
test_df_selected = test_df[selected_features]

In [ ]:
X_train = train_df_selected.values
X_test = test_df_selected.values
y_train = y_train.values
y_test = y_test.values

In [ ]:

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 3. Model Implementation

### 3.1 Ordinal Logistic Regression

In [ ]:
class OrdinalLogisticRegression():
    # A dummy hyperparameter is put as a placeholder for now
    def __init__(self, max_iter=100, C=2.0, solver='liblinear', tol=1e-4, penalty='l2'):
        self.max_iter = max_iter
        self.C = C
        self.solver = solver
        self.tol = tol
        self.penalty = penalty

        self.classes_ = []
        self.models_ = []


    def fit(self, X, y):
        self.classes_ = sorted(np.unique(y))
        self.models_ = []

        # Train k-1 binary logistic regression models
        for i, c in enumerate(self.classes_[:-1]):  # We need only k-1 classifiers
            # Step 1: Create binary labels (0 or 1) for the i-th model
            y_i = np.where(y <= c, 0, 1)

            # Step 2: Initialize the logistic regression model with parameters
            model = LogisticRegression(max_iter=self.max_iter, C=self.C, solver=self.solver, tol=self.tol, penalty=self.penalty)

            # Step 3: Fit the model with the features and the binary labels
            model.fit(X, y_i)

            # Step 4: Append the fitted model to the list of models
            self.models_.append(model)

        return self



    def predict_proba(self, X):
        assert len(self.models_) > 0, "Model is not fitted yet. Run .fit() first."

        # Predicted probabilities for k-1 binary logistic regression models (initialize placeholders)
        binary_probabilities = np.empty((X.shape[0], len(self.models_), 2), dtype=float)

        # Make predictions of k-1 binary logistic regression models
        for i, model in enumerate(self.models_):
            # Predicted probabilities by the i-th binary logistic regression model
            binary_probabilities[:, i] = model.predict_proba(X)

        # Compute the probabilities to be in each class
        k = len(self.classes_)
        proba = np.empty((X.shape[0], k), dtype=float)  # predicted probabilities for all the data in X (shape=(X.shape[0], k))

        # Set the probability for the 0th class
        proba[:, 0] = binary_probabilities[:, 0, 0]

        # Compute the intermediate probabilities for each class
        for i in range(1, k - 1):
            proba[:, i] = binary_probabilities[:, i, 0] - binary_probabilities[:, i - 1, 0]

        # Set the probability for the last class
        proba[:, -1] = binary_probabilities[:, k - 2, 1]

        # Ensure that each row sums to 1
        assert np.allclose(proba.sum(axis=1), 1), 'There is a problem in the probability computation'

        return proba


    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis = 1)

    # Add hyperparameters here whenever you add new ones
    # max_iter is added as an example here. key is the hyperparameter name,
    # and value is the attribute name you defined in this class
    def get_params(self, deep=True):
        return {'max_iter': self.max_iter}

    # DO NOT CHANGE
    def set_params(self, **parameters):
        # Set estimator parameters
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

### 3.2 K-Fold Cross-Validation

### 3.3 Bias-Variance Analysis

In [ ]:
### NOTE: You don't need to change anything in this code block! ###

def _draw_bootstrap_sample(rng, X, y):
    sample_indices = np.arange(X.shape[0])
    bootstrap_indices = rng.choice(
        sample_indices, size=sample_indices.shape[0], replace=True
    )
    return X[bootstrap_indices], y[bootstrap_indices]

def bias_variance_decomp(
    estimator,
    X_train,
    y_train,
    X_test,
    y_test,
    num_rounds=10,
    random_seed=0
):
    """
    estimator : object
        A classifier or regressor object or class implementing both a
        `fit` and `predict` method similar to the scikit-learn API.

    X_train : array-like, shape=(num_examples, num_features)
        A training dataset for drawing the bootstrap samples to carry
        out the bias-variance decomposition.

    y_train : array-like, shape=(num_examples)
        Targets (class labels, continuous values in case of regression)
        associated with the `X_train` examples.

    X_test : array-like, shape=(num_examples, num_features)
        The test dataset for computing the average loss, bias,
        and variance.

    y_test : array-like, shape=(num_examples)
        Targets (class labels, continuous values in case of regression)
        associated with the `X_test` examples.

    num_rounds : int (default=10)
        Number of bootstrap rounds (sampling from the training set)
        for performing the bias-variance decomposition. Each bootstrap
        sample has the same size as the original training set.

    random_seed : int (default=0)
        Random seed for the bootstrap sampling used for the
        bias-variance decomposition.

    Returns
    ----------
    avg_bias, avg_var : returns the average bias, and average bias (all floats),
                        where the average is computed over the data points
                        in the test set.

    """
    loss = "mse"

    for ary in (X_train, y_train, X_test, y_test):
        assert type(ary) == np.ndarray, \
            "X_train, y_train, X_test, y_test have to be NumPy array. \
            If e.g., X_train is a pandas DataFrame, convert it to NumPy array \
            via X_train=X_train.values."

    rng = np.random.RandomState(random_seed)

    # All the predictions across different rounds
    all_pred = np.zeros((num_rounds, y_test.shape[0]), dtype=np.float64)

    for i in range(num_rounds):
        # Randomly sample training data
        X_boot, y_boot = _draw_bootstrap_sample(rng, X_train, y_train)

        # Fit the model using the randomly sampled data
        pred = estimator.fit(X_boot, y_boot).predict(X_test)
        all_pred[i] = pred

    # Mean prediction across runs using different dataset for each data point
    main_predictions = np.mean(all_pred, axis=0)

    # Average bias across different rounds
    avg_bias = np.sum((main_predictions - y_test) ** 2) / y_test.size

    # Average variance across different rounds
    avg_var = np.sum((main_predictions - all_pred) ** 2) / all_pred.size

    return avg_bias, avg_var

In [ ]:
# Usage example
model = OrdinalLogisticRegression(max_iter=1000, C=2.0, solver='liblinear', tol=1e-4, penalty='l1')
avg_bias, avg_var = \
    bias_variance_decomp(model, X_train, y_train, X_test, y_test, num_rounds=10, random_seed=0)

In [ ]:
print("Average Bias:", avg_bias)
print("Average Variance:", avg_var)

**Average Bias (~20.02):**

This bias is relatively high. This suggests that, on average, the model's predictions deviate significantly from the actual target values in the test set. High bias typically implies that the model is underfitting, i.e., it may not have learned the underlying patterns in the data effectively.

**Average Variance (~1.61):**

This variance is relatively low. This indicates that the model's predictions are fairly stable across different training samples. Low variance suggests that the model is not highly sensitive to the specific samples it was trained on, which is usually a good sign.

### 3.4 Feature Scaling

In [ ]:
# Initialize scaler
scaler = StandardScaler()

# Apply scaling to training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to testing data
X_test_scaled = scaler.transform(X_test)

## 4. Hyperparameter Tuning

In [ ]:
# Define the model
model = OrdinalLogisticRegression(max_iter=100, C=2.0, solver='liblinear', tol=1e-4, penalty='l2')  # Initialize with default or baseline parameters

# Define the parameter grid for OrdinalLogisticRegression
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],           # Range for regularization strength
    'penalty': ['l1', 'l2']                 # L1 and L2 regularization
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,                # Reduce to 3-fold cross-validation
    verbose=2,
    n_jobs=-1
)

# Perform the grid search
grid_search.fit(X_train, y_train)

# Display the best parameters and accuracy score from the grid search
print("Best parameters found:", grid_search.best_params_)
print("Best cross-validated accuracy:", grid_search.best_score_)

# Use the best model to make predictions on the test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Evaluate the model with accuracy, precision, recall, and F1-score
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, average='weighted')  # Use 'weighted' for multi-class
test_recall = recall_score(y_test, y_pred, average='weighted')        # Same here for recall
test_f1 = f1_score(y_test, y_pred, average='weighted')                # Use 'weighted' for multi-class

# Display evaluation metrics
print("Test accuracy with best model:", test_accuracy)
print("Test precision with best model:", test_precision)
print("Test recall with best model:", test_recall)
print("Test F1-score with best model:", test_f1)

### 4.1 Feature Importance

In [ ]:
model = OrdinalLogisticRegression()
model = model.fit(X_train, y_train)


# Assuming these are the best hyperparameters identified via grid search
best_max_iter = 100
best_C = 0.1
best_solver = 'liblinear'
best_tol = 0.0001
best_penalty = 'l2'

# Initialize the model with the best hyperparameters
model = OrdinalLogisticRegression(
    max_iter=best_max_iter,
    C=best_C,
    solver=best_solver,
    tol=best_tol,
    penalty=best_penalty
)

# Fit the model with training data
model = model.fit(X_train, y_train)

In [ ]:
# Define feature names manually if X_train is a NumPy array
feature_names = ['Q4', 'Q2', 'Q27', 'Q35_1_nan', 'Q12_10_nan', 'Q36_2_nan', 'Q36_3_nan', 'Q24', 'Q12_11_nan', 'Q30',
                 'Q12_4_nan', 'Q36_11_nan', 'Q37_8_nan', 'Q12_7_nan', 'Q42_1_nan', 'Q35_13_nan', 'Q39_3_nan', 'Q11',
                 'Q36_9_nan', 'Q12_2_nan', 'Q35_16_nan', 'Q12_3_nan', 'Q35_10_nan', 'Experience_Education', 'Q35_6_nan',
                 'Q12_9_nan', 'Q34_3_nan']  # Add your feature names here

# Access the coefficients from the first binary logistic regression model in the sequence
coefficients = model.models_[0].coef_[0]

# Create a DataFrame for visualization
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort features by ordered value of coefficients to show the most important ones from left to right on the x-axis
feature_importance_df['ord_Coefficient'] = feature_importance_df['Coefficient']
feature_importance_df = feature_importance_df.sort_values(by='ord_Coefficient', ascending=False)

# Plot
plt.figure(figsize=(12, 8))
plt.bar(feature_importance_df['Feature'], feature_importance_df['Coefficient'], color='teal')
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
plt.ylabel('Coefficient')
plt.xlabel('Feature')
plt.title('Feature Importance Based on Ordinal Logistic Regression Coefficients')
plt.tight_layout()  # Adjust layout to fit everything nicely
plt.show()



# Sort features by absolute value of coefficients for the additional plot
feature_importance_df['Abs_Coefficient'] = feature_importance_df['Coefficient'].abs()
feature_importance_df_sorted = feature_importance_df.sort_values(by='Abs_Coefficient', ascending=False)

# New plot (features ordered by absolute value of coefficient)
plt.figure(figsize=(12, 6))
plt.bar(feature_importance_df_sorted['Feature'], feature_importance_df_sorted['Abs_Coefficient'], color='coral')
plt.ylabel('Absolute Coefficient')
plt.xlabel('Feature')
plt.title('Feature Importance Ordered by Absolute Value of Coefficients')
plt.xticks(rotation=90)  # Rotate x labels for better readability
plt.show()

## 5. Final Evaluation

In [ ]:
# Define the model with the best hyperparameters
model = OrdinalLogisticRegression(
    max_iter=1000,  # Example value, replace with best value found
    C=0.1,          # Example value, replace with best value found
    solver='liblinear',  # Example value, replace with best value found
    tol=1e-4,       # Example value, replace with best value found
    penalty='l1'    # Example value, replace with best value found
)

# Fit the model on the training set
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score, classification_report

# Evaluate on the training set
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", train_accuracy)
print("Training Classification Report:")
print(classification_report(y_train, y_train_pred, digits=3))

# Evaluate on the test set
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy:", test_accuracy)
print("Test Classification Report:")
print(classification_report(y_test, y_test_pred, digits=3))



In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Plot distribution for the training set
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(y_train, color='blue', kde=True, label='True Values', bins=len(set(y_train)))
sns.histplot(y_train_pred, color='orange', kde=True, label='Predictions', bins=len(set(y_train)))
plt.title("Training Set: True vs Predicted Distribution")
plt.xlabel("Target Variable")
plt.legend()

# Plot distribution for the test set
plt.subplot(1, 2, 2)
sns.histplot(y_test, color='blue', kde=True, label='True Values', bins=len(set(y_test)))
sns.histplot(y_test_pred, color='orange', kde=True, label='Predictions', bins=len(set(y_test)))
plt.title("Test Set: True vs Predicted Distribution")
plt.xlabel("Target Variable")
plt.legend()

plt.tight_layout()
plt.show()

## Conclusions

**Pipeline summary:** 72 survey features → Chi-square filtering → LassoCV selection → ordinal logistic regression (C=2.0, solver='lbfgs') → GridSearchCV tuning.

**Key findings:**
- LassoCV selected ~15 features from 72; Q4 (education), Q2 (age), and Q27 (experience) are the strongest predictors
- High bias (~20) reflects the difficulty of ordinal salary prediction from survey data — the compensation buckets span a wide range and features are noisy
- GridSearchCV improved accuracy; confusion matrix shows most errors are in adjacent salary buckets (not extreme misclassifications)
- The dataset's heavy skew toward low-compensation respondents (class imbalance) limits precision on higher salary classes